# KG1 v87 - Submit Kaggle do checkpoint-600

Use este notebook em um Colab separado, sem parar o treino atual.

Ele faz:

1. monta o Google Drive;
2. instala/configura Kaggle CLI;
3. valida o checkpoint-600;
4. cria `submission_v87_checkpoint600.zip` com apenas `adapter_config.json` e `adapter_model.safetensors`;
5. envia para a competicao `nvidia-nemotron-model-reasoning-challenge`.

O notebook nao grava nem imprime o conteudo do `kaggle.json`; ele apenas pede upload se a credencial ainda nao existir no runtime.

In [ ]:
# KG1 v87 checkpoint-600 Kaggle submit
# Rode em um Colab separado do treino.

!pip -q install kaggle

from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import csv
import io
import json
import os
import shutil
import subprocess
import zipfile

# ===== CONFIG =====
COMPETITION = "nvidia-nemotron-model-reasoning-challenge"
ADAPTER_DIR = Path(
    "/content/drive/MyDrive/KG1_NVIDIA_Nemotron_v87/"
    "colab_runs/v87-colab-h100-20260416-230931/"
    "training_output_fastlogs_mlen4096_bsz16/checkpoint-600"
)
ZIP_PATH = Path("/content/submission_v87_checkpoint600.zip")
MESSAGE = "v87 checkpoint-600 eval_loss 1.8460"

# Se quiser apenas validar e gerar o ZIP sem submeter, mude para False.
RUN_SUBMIT = True

# Kaggle costuma limitar a 5 submits/dia. Use True apenas se quiser ignorar essa checagem.
MAX_DAILY_SUBMITS = 5
SKIP_SLOT_CHECK = False


def run_cmd(cmd, check=True):
    print("\n$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def configure_kaggle():
    kaggle_dir = Path("/root/.kaggle")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    cred_path = kaggle_dir / "kaggle.json"

    if not cred_path.exists():
        print("Upload do kaggle.json agora.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("O upload precisa conter um arquivo chamado kaggle.json")
        shutil.copy("kaggle.json", cred_path)

    os.chmod(cred_path, 0o600)
    print(f"Kaggle configurado em {cred_path}")


def validate_adapter(adapter_dir: Path):
    required = ["adapter_config.json", "adapter_model.safetensors"]

    print("\n=== VALIDANDO CHECKPOINT ===")
    print("Adapter dir:", adapter_dir)
    print("Existe:", adapter_dir.exists())
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Checkpoint nao encontrado: {adapter_dir}")

    for name in required:
        path = adapter_dir / name
        if not path.exists():
            raise FileNotFoundError(f"Arquivo obrigatorio ausente: {path}")
        print(f"OK {name}: {path.stat().st_size / 1e6:.1f} MB")

    cfg = json.loads((adapter_dir / "adapter_config.json").read_text())
    rank = int(cfg.get("r", cfg.get("lora_rank", 999)))
    targets = cfg.get("target_modules", [])
    if isinstance(targets, str):
        targets = [targets]

    print("LoRA rank:", rank)
    print("target_modules:", targets)

    if rank > 32:
        raise ValueError(f"Rank {rank} > 32. Nao submeter.")
    if "in_proj" not in targets:
        raise ValueError("target_modules nao contem in_proj. Nao submeter.")
    for bad in ["gate_proj", "x_proj"]:
        if bad in targets:
            raise ValueError(f"target_modules contem {bad}. Nao submeter.")

    print("Gate do adapter: OK")


def create_submission_zip(adapter_dir: Path, zip_path: Path):
    print("\n=== GERANDO ZIP ===")
    if zip_path.exists():
        zip_path.unlink()

    # ZIP_STORED evita gastar tempo tentando comprimir safetensors, que quase nao reduz.
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for name in ["adapter_config.json", "adapter_model.safetensors"]:
            zf.write(adapter_dir / name, arcname=name)
            print("Adicionado:", name)

    print("ZIP criado:", zip_path)
    print("Tamanho:", round(zip_path.stat().st_size / 1e6, 1), "MB")
    return zip_path


def check_submit_slots():
    print("\n=== SUBMISSOES RECENTES ===")
    result = run_cmd(
        ["kaggle", "competitions", "submissions", "-c", COMPETITION, "--csv"],
        check=False,
    )
    if result.returncode != 0:
        print("Nao consegui checar slots. Continuando sem bloquear.")
        return

    rows = list(csv.DictReader(io.StringIO(result.stdout)))
    today_utc = datetime.now(timezone.utc).date().isoformat()
    today_count = sum(str(row.get("date", ""))[:10] == today_utc for row in rows)
    print(f"Submits hoje UTC: {today_count}/{MAX_DAILY_SUBMITS}")

    for row in rows[:5]:
        print({
            "date": row.get("date"),
            "status": row.get("status"),
            "publicScore": row.get("publicScore"),
            "description": row.get("description"),
        })

    if not SKIP_SLOT_CHECK and today_count >= MAX_DAILY_SUBMITS:
        raise RuntimeError("Limite diario de submits aparentemente esgotado.")


def submit(zip_path: Path):
    print("\n=== SUBMIT ===")
    if not RUN_SUBMIT:
        print("RUN_SUBMIT=False. Validacao completa, mas nenhum submit foi enviado.")
        print("ZIP pronto em:", zip_path)
        return

    run_cmd([
        "kaggle", "competitions", "submit",
        "-c", COMPETITION,
        "-f", str(zip_path),
        "-m", MESSAGE,
    ])

    print("\nSubmit enviado.")
    print(f"Acompanhe em: https://www.kaggle.com/competitions/{COMPETITION}/submissions")


# ===== EXECUCAO =====
drive.mount("/content/drive")
configure_kaggle()
validate_adapter(ADAPTER_DIR)
submission_zip = create_submission_zip(ADAPTER_DIR, ZIP_PATH)
check_submit_slots()
submit(submission_zip)


## Checar score depois

Quando o Kaggle terminar de processar, rode a celula abaixo para listar as ultimas submissoes.

In [ ]:
!kaggle competitions submissions -c nvidia-nemotron-model-reasoning-challenge | head -20
